<a href="https://colab.research.google.com/github/EmePin/Analisis-de-datos/blob/main/Fake_News_Prediction_v1_0_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# Instalar dependencias necesarias
!pip install gradio transformers torch scikit-learn pandas numpy kagglehub
!pip install huggingface_hub

# Importar librerías
import os
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import pickle
import gradio as gr
import kagglehub

In [12]:
# Descargar el dataset
print("Descargando dataset...")
path = kagglehub.dataset_download("subho117/fake-news-detection-using-machine-learning")
print(f"Dataset descargado en: {path}")

# Buscar archivos CSV en el directorio descargado
def find_csv_files(directory):
    csv_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.csv'):
                csv_files.append(os.path.join(root, file))
    return csv_files

csv_files = find_csv_files(path)
print("Archivos CSV encontrados:")
for file in csv_files:
    print(f"  - {file}")

# Cargar el dataset (ajusta el nombre del archivo según lo que encuentres)
try:
    # Intentar cargar diferentes posibles nombres de archivo
    possible_files = ['train.csv', 'data.csv', 'fake_news.csv', 'FakeNews.csv']
    df = None

    for csv_file in csv_files:
        filename = os.path.basename(csv_file).lower()
        if any(possible in filename for possible in possible_files):
            print(f"Cargando: {csv_file}")
            df = pd.read_csv(csv_file)
            break

    if df is None and csv_files:
        # Si no encuentra con nombres específicos, cargar el primero
        print(f"Cargando el primer archivo encontrado: {csv_files[0]}")
        df = pd.read_csv(csv_files[0])

    print(f"Dataset cargado. Tamaño: {df.shape}")
    print(f"Columnas: {df.columns.tolist()}")

    # Mostrar las primeras filas
    print("\nPrimeras 5 filas:")
    print(df.head())

except Exception as e:
    print(f"Error al cargar el dataset: {e}")
    # Crear datos de ejemplo si hay problemas
    print("Creando datos de ejemplo...")
    data = {
        'text': [
            'Breaking: Scientists discover new planet that could support life',
            'The government is hiding alien technology from the public',
            'New study shows benefits of daily exercise on mental health',
            'Celebrity claims drinking bleach cures all diseases',
            'Economists predict steady growth for next quarter',
            'Secret elite group controls world governments behind scenes'
        ],
        'label': [1, 0, 1, 0, 1, 0]  # 1 = real, 0 = fake
    }
    df = pd.DataFrame(data)

Descargando dataset...
Using Colab cache for faster access to the 'fake-news-detection-using-machine-learning' dataset.
Dataset descargado en: /kaggle/input/fake-news-detection-using-machine-learning
Archivos CSV encontrados:
  - /kaggle/input/fake-news-detection-using-machine-learning/News.csv
Cargando el primer archivo encontrado: /kaggle/input/fake-news-detection-using-machine-learning/News.csv
Dataset cargado. Tamaño: (44919, 6)
Columnas: ['Unnamed: 0', 'title', 'text', 'subject', 'date', 'class']

Primeras 5 filas:
   Unnamed: 0                                              title  \
0           0   Donald Trump Sends Out Embarrassing New Year’...   
1           1   Drunk Bragging Trump Staffer Started Russian ...   
2           2   Sheriff David Clarke Becomes An Internet Joke...   
3           3   Trump Is So Obsessed He Even Has Obama’s Name...   
4           4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0

In [13]:
# Preparar los datos
print("\nPreparando datos para el modelo...")

# Verificar las columnas disponibles
print("Columnas disponibles:", df.columns.tolist())

# Buscar columnas que puedan contener texto y etiquetas
text_column = None
label_column = None

# Posibles nombres para columnas de texto
possible_text_cols = ['text', 'content', 'article', 'news', 'title', 'headline', 'statement']
# Posibles nombres para columnas de etiqueta
possible_label_cols = ['label', 'target', 'class', 'is_fake', 'fake']

for col in df.columns:
    col_lower = col.lower()
    if any(text in col_lower for text in possible_text_cols) and text_column is None:
        text_column = col
    if any(label in col_lower for label in possible_label_cols) and label_column is None:
        label_column = col

# Si no encontramos columnas con nombres esperados, usar las primeras
if text_column is None:
    # Buscar columnas con tipo de dato string/object
    for col in df.columns:
        if df[col].dtype == 'object' and text_column is None:
            text_column = col
            break

if label_column is None:
    # Buscar columnas con tipo numérico o booleano
    for col in df.columns:
        if col != text_column and (df[col].dtype in ['int64', 'float64', 'bool']):
            label_column = col
            break

print(f"Usando columna de texto: '{text_column}'")
print(f"Usando columna de etiqueta: '{label_column}'")

# Limpiar datos nulos
df_clean = df.dropna(subset=[text_column])
if label_column:
    df_clean = df_clean.dropna(subset=[label_column])

# Preparar X e y
X = df_clean[text_column].astype(str).tolist()

if label_column:
    y = df_clean[label_column].astype(int).tolist()
else:
    # Si no hay etiquetas, crear unas ficticias para demostración
    print("No se encontró columna de etiquetas. Creando etiquetas de ejemplo...")
    y = [1, 0, 1, 0, 1, 0] * (len(X) // 6 + 1)
    y = y[:len(X)]

print(f"Datos preparados: {len(X)} ejemplos")

# Vectorizar el texto
print("\nVectorizando texto...")
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_vectorized = vectorizer.fit_transform(X)

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X_vectorized, y, test_size=0.2, random_state=42, stratify=y
)

# Entrenar modelo
print("Entrenando modelo...")
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Evaluar modelo
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Precisión del modelo: {accuracy:.4f}")

# Guardar modelo y vectorizador
print("\nGuardando modelo y vectorizador...")
with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Modelo entrenado y guardado exitosamente!")


Preparando datos para el modelo...
Columnas disponibles: ['Unnamed: 0', 'title', 'text', 'subject', 'date', 'class']
Usando columna de texto: 'title'
Usando columna de etiqueta: 'class'
Datos preparados: 44919 ejemplos

Vectorizando texto...
Entrenando modelo...
Precisión del modelo: 0.9440

Guardando modelo y vectorizador...
Modelo entrenado y guardado exitosamente!


In [14]:
# Función para predecir si una noticia es falsa o real
def predict_news(text):
    """
    Predice si una noticia es falsa o real
    Retorna: (etiqueta, probabilidad)
    """
    try:
        # Vectorizar el texto de entrada
        text_vectorized = vectorizer.transform([text])

        # Predecir
        prediction = model.predict(text_vectorized)[0]
        probability = model.predict_proba(text_vectorized)[0]

        # Interpretar resultado
        # Asumimos: 1 = real, 0 = fake
        label = "REAL" if prediction == 1 else "FALSA"

        # Obtener probabilidad para la clase predicha
        prob = probability[prediction] * 100

        # Crear explicación
        explanation = ""
        if prediction == 1:
            explanation = f"Esta noticia parece REAL con un {prob:.1f}% de confianza."
        else:
            explanation = f"Esta noticia parece FALSA con un {prob:.1f}% de confianza."

        # Detalles adicionales
        details = f"""
        **Análisis detallado:**
        - Probabilidad de ser REAL: {probability[1]*100:.1f}%
        - Probabilidad de ser FALSA: {probability[0]*100:.1f}%
        - Longitud del texto: {len(text)} caracteres
        - Palabras clave detectadas: {len(text.split())} palabras
        """

        return label, explanation, details

    except Exception as e:
        return "ERROR", f"Error al procesar: {str(e)}", ""

In [18]:
# Crear la aplicación Gradio
def create_gradio_app():
    with gr.Blocks(theme=gr.themes.Soft(), title="Detector de Noticias Falsas") as app:
        gr.Markdown("# 📰 Detector de Noticias Falsas")
        gr.Markdown("""
        Esta aplicación utiliza inteligencia artificial para analizar noticias y determinar
        si son **reales** o **falsas**.

        ### Cómo usar:
        1. Escribe o pega una noticia en el cuadro de texto
        2. Haz clic en "Analizar Noticia"
        3. Revisa los resultados y el análisis detallado

        ⚠️ **Nota:** Esta es una herramienta de apoyo. Siempre verifica la información con fuentes confiables.
        """)

        with gr.Row():
            with gr.Column(scale=2):
                news_input = gr.Textbox(
                    label="Ingresa la noticia a analizar",
                    placeholder="Escribe o pega aquí el texto de la noticia...",
                    lines=10,
                    max_lines=20
                )

                analyze_btn = gr.Button("🔍 Analizar Noticia", variant="primary", size="lg")

                # Ejemplos
                gr.Markdown("### 📋 Ejemplos para probar:")
                examples = [
                    ["Los científicos descubren nueva terapia para el cáncer con 95% de efectividad según estudio publicado en Nature."],
                    ["El gobierno oculta la existencia de extraterrestres que viven entre nosotros según documentos filtados."],
                    ["La bolsa de valores alcanza máximo histórico impulsada por avances tecnológicos."],
                    ["Celebridad revela que beber agua de mar cura todas las enfermedades sin base científica."]
                ]
                gr.Examples(examples=examples, inputs=news_input)

            with gr.Column(scale=1):
                # Resultados
                result_label = gr.Label(
                    label="Resultado",
                    value="Esperando análisis...",
                    color="grey"
                )

                result_output = gr.Textbox(
                    label="Explicación",
                    interactive=False,
                    lines=4
                )

                # Gráfico de probabilidades
                with gr.Accordion("📊 Análisis Detallado", open=False):
                    details_output = gr.Markdown("Los detalles aparecerán aquí después del análisis.")

        # Consejos
        with gr.Accordion("💡 Consejos para identificar noticias falsas", open=False):
            gr.Markdown("""
            1. **Verifica la fuente**: ¿Es un medio confiable y conocido?
            2. **Busca el autor**: ¿Existe? ¿Es creíble?
            3. **Comprueba la fecha**: ¿La noticia es actual o antigua?
            4. **Consulta múltiples fuentes**: ¿Otros medios respetables reportan lo mismo?
            5. **Analiza el lenguaje**: ¿Es sensacionalista o emocional?
            6. **Revisa la evidencia**: ¿Hay datos, estudios o testimonios verificables?
            7. **Usa fact-checkers**: Sitios como Snopes, FactCheck.org, Maldita.es
            """)

        # Funcionalidad del botón
        analyze_btn.click(
            fn=predict_news,
            inputs=news_input,
            outputs=[result_label, result_output, details_output]
        )

        # También permitir Enter para enviar
        news_input.submit(
            fn=predict_news,
            inputs=news_input,
            outputs=[result_label, result_output, details_output]
        )

    return app

# Crear y lanzar la aplicación
print("\nCreando aplicación Gradio...")
app = create_gradio_app()

# Para ejecutar en Colab
print("¡Aplicación creada exitosamente!")
print("\nPara usar la aplicación en Colab, ejecuta:")
print("app.launch(share=True)")


Creando aplicación Gradio...
¡Aplicación creada exitosamente!

Para usar la aplicación en Colab, ejecuta:
app.launch(share=True)


In [19]:
# Create the Gradio application
def create_gradio_app():
    with gr.Blocks(theme=gr.themes.Soft(), title="Fake News Detector") as app:
        gr.Markdown("# 📰 Fake News Detector")
        gr.Markdown("""
        This application uses artificial intelligence to analyze news articles and determine
        whether they are **real** or **fake**.

        ### How to use:
        1. Write or paste a news article into the text box
        2. Click "Analyze News"
        3. Review the results and the detailed analysis

        ⚠️ **Note:** This is a support tool. Always verify information with reliable sources.
        """)

        with gr.Row():
            with gr.Column(scale=2):
                news_input = gr.Textbox(
                    label="Enter the news article to analyze",
                    placeholder="Write or paste the news article text here...",
                    lines=10,
                    max_lines=20
                )

                analyze_btn = gr.Button("🔍 Analyze News", variant="primary", size="lg")

                # Examples
                gr.Markdown("### 📋 Examples to try:")
                examples = [
                    ["Scientists discover a new cancer therapy with 95% effectiveness according to a study published in Nature."],
                    ["The government hides the existence of aliens living among us according to leaked documents."],
                    ["The stock market reaches an all-time high driven by technological advances."],
                    ["A celebrity reveals that drinking seawater cures all diseases without scientific basis."]
                ]
                gr.Examples(examples=examples, inputs=news_input)

            with gr.Column(scale=1):
                # Results
                result_label = gr.Label(
                    label="Result",
                    value="Waiting for analysis...",
                    color="grey"
                )

                result_output = gr.Textbox(
                    label="Explanation",
                    interactive=False,
                    lines=4
                )

                # Probability chart
                with gr.Accordion("📊 Detailed Analysis", open=False):
                    details_output = gr.Markdown("Details will appear here after the analysis.")

        # Tips
        with gr.Accordion("💡 Tips for identifying fake news", open=False):
            gr.Markdown("""
            1. **Verify the source**: Is it a reliable and well-known media outlet?
            2. **Look for the author**: Do they exist? Are they credible?
            3. **Check the date**: Is the news current or old?
            4. **Consult multiple sources**: Do other reputable media report the same?
            5. **Analyze the language**: Is it sensationalist or overly emotional?
            6. **Check the evidence**: Are there verifiable data, studies, or testimonials?
            7. **Use fact-checkers**: Sites like Snopes, FactCheck.org, Maldita.es
            """)

        # Button functionality
        analyze_btn.click(
            fn=predict_news,
            inputs=news_input,
            outputs=[result_label, result_output, details_output]
        )

        # Also allow Enter to submit
        news_input.submit(
            fn=predict_news,
            inputs=news_input,
            outputs=[result_label, result_output, details_output]
        )

    return app

# Create and launch the application
print("\nCreating Gradio application...")
app = create_gradio_app()

# To run in Colab
print("Application created successfully!")
print("\nTo use the application in Colab, run:")
print("app.launch(share=True)")



Creating Gradio application...
Application created successfully!

To use the application in Colab, run:
app.launch(share=True)


In [20]:
# Lanzar la aplicación en Colab
try:
    # Esto lanzará la aplicación en Colab
    app.launch(share=True)
except:
    # Si hay error, lanzar localmente
    app.launch(debug=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0af07219f13c358a66.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
